In [72]:
import pandas as pd
import html
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('./data/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns=['label', 'message']
df['label'] = df['label'].map({'ham':0, 'spam':1})
def clean_message(msg):
    msg = html.unescape(str(msg))
    msg = re.sub(r"<#>", " ", msg)
    msg = re.sub(r"<\d+>", " ", msg)
    msg = re.sub(r"\s+", " ", msg)
    return msg
df['message_clean'] = df['message'].apply(clean_message)

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words='english', max_features=10000, ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=1000, class_weight='balanced'))
])
X_train, X_test, y_train, y_test = train_test_split(df['message_clean'], df['label'], random_state=42, stratify=df['label'])
model.fit(X_train, y_train)

pred = model.predict(X_test)



In [73]:
pred = model.predict(X_test)
print(accuracy_score(y_test, pred))
print("------------------------------------------------------")
print(confusion_matrix(y_test, pred, labels=[0, 1]))
print("------------------------------------------------------")
print(classification_report(y_test, pred))

0.9827709978463748
------------------------------------------------------
[[1195   11]
 [  13  174]]
------------------------------------------------------
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1206
           1       0.94      0.93      0.94       187

    accuracy                           0.98      1393
   macro avg       0.96      0.96      0.96      1393
weighted avg       0.98      0.98      0.98      1393



In [75]:
import joblib

joblib.dump(model, "spam_classifier.joblib")


['spam_classifier.joblib']

In [76]:
loaded = joblib.load("spam_classifier.joblib")


In [79]:
messages = [
    "Congratulations! You have won a free iPhone. Click here now!",
    "Hey, are we still meeting at 6?",
    "URGENT! Your bank account has been locked. Verify immediately.",
    "Can you send me the report by tomorrow?"
]

preds = loaded.predict(messages)
for message, pred in zip(messages, preds):
    print("message: ", message, "result: ", "spam" if pred==1 else "ham")

message:  Congratulations! You have won a free iPhone. Click here now! result:  spam
message:  Hey, are we still meeting at 6? result:  ham
message:  URGENT! Your bank account has been locked. Verify immediately. result:  spam
message:  Can you send me the report by tomorrow? result:  ham


In [86]:
for message, pred in zip(messages, loaded.predict_proba(messages)):
    print("message: ", message, "ham: ", pred[0], "spam: ", pred[1], "classify: " , "true" if pred[1] >= 0.5 else "false")

message:  Congratulations! You have won a free iPhone. Click here now! ham:  0.14401168881602366 spam:  0.8559883111839763 classify:  true
message:  Hey, are we still meeting at 6? ham:  0.9368213760076025 spam:  0.06317862399239749 classify:  false
message:  URGENT! Your bank account has been locked. Verify immediately. ham:  0.3634123044958636 spam:  0.6365876955041364 classify:  true
message:  Can you send me the report by tomorrow? ham:  0.7919493506940534 spam:  0.2080506493059467 classify:  false


array([0, 1])